# Project 2 — Veggie Classification with PyTorch

In this project you will classify images of vegetables into 10 categories using PyTorch.

You must build **two models**:
1. **A CNN from scratch** — design and train your own convolutional neural network
2. **A fine-tuned pretrained model** — take a model pretrained on ImageNet and adapt it for this task

## Deliverables

Submit the following to BrighSpace:
- Your github repository link
- Your saved model file (`.pth`) — must be under 400 MB
- A note (~1-2 paragraphs) explaining what you did to improve accuracy beyond a basic model

## Grading

| Component | Weight |
|---|---|
| Accuracy (validation set) | 60% |
| Code — readable and logical | 20% |
| Explanatory note | 20% |

In [ ]:
from getpass import getpass

# 1. Paste the FULL URL from your browser address bar here
# Example: https://github.com/org-name/repo-name
full_url = "https://github.com/3950-1252-Intro-to-machine-learning/project2-nnimage-data-3950-team-1"

# 2. This part cleans the URL to add your token automatically
token = getpass('Enter your GitHub Token: ')
authenticated_url = full_url.replace("https://", f"https://{token}@") + ".git"

# 3. Clone it
!git clone {authenticated_url}

In [ ]:
import os
import shutil

# The folder that was just created
repo_name = "project2-nnimage-data-3950-team-1"

if os.path.exists(repo_name):
    for file in os.listdir(repo_name):
        # We move everything except the hidden .git folder
        if file != ".git":
            src = os.path.join(repo_name, file)
            dst = os.path.join(".", file)

            # This handles moving files and folders
            if not os.path.exists(dst):
                shutil.move(src, dst)
    print("✅ Done! utils.py and other files are now in the main directory.")
else:
    print("❌ Folder not found. Check the sidebar to see what the folder name is.")

In [ ]:
# Imports
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torch.optim import lr_scheduler
import torch.nn.functional as F

from utils import get_device, show_images, plot_training_history, evaluate, per_class_accuracy

device = get_device()
print(f"Using device: {device}")

## Load the Data

Download `Vegetables.zip` from Moodle and place it in your working directory.

**On Colab**: You can drag-and-drop the zip into the file browser, or mount your Google Drive (see below).

The dataset contains:
- **Training**: 20,000 images (10 classes)
- **Validation**: 4,000 images (10 classes)

#### Temporary Files Warning
If you get errors loading images, remove temp files (e.g., `._` files on Mac):
- Mac: `dot_clean -n .` in the image folder
- Windows: search for and delete `._*` files

In [ ]:
# Optional: Load from Google Drive on Colab
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "/content/drive/My Drive/Vegetables.zip" "Vegetables.zip"

In [ ]:
# Unzip the dataset
zip_name = "Vegetables.zip"

if not os.path.exists("Vegetables"):
    with zipfile.ZipFile(zip_name, "r") as zip_ref:
        zip_ref.extractall()
    print("Extracted Vegetables.zip")
else:
    print("Vegetables folder already exists")

In [ ]:
# Data directories
train_dir = "Vegetables/train"
val_dir = "Vegetables/validation"

IMAGE_SIZE = 224

# Define transforms — you may modify these to improve accuracy!
transform_train = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

transform_val = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# Load datasets using ImageFolder (reads class names from subfolder names)
train_dataset = ImageFolder(train_dir, transform=transform_train)
val_dataset = ImageFolder(val_dir, transform=transform_val)

class_names = train_dataset.classes

print(f"Training samples:   {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Classes ({len(class_names)}): {class_names}")

In [ ]:
# Create data loaders
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# Visualize some training images
viz_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])
viz_dataset = ImageFolder(train_dir, transform=viz_transform)
indices = torch.randperm(len(viz_dataset))[:8]
viz_images = torch.stack([viz_dataset[i][0] for i in indices])
viz_labels = [viz_dataset[i][1] for i in indices]
show_images(viz_images, viz_labels, class_names)

---

## Part 1: Build a CNN from Scratch

Design your own CNN architecture. Some things to consider:
- How many convolutional layers/blocks do you need?
- What filter sizes and counts will you use?
- Will you use batch normalization? Dropout?
- What pooling strategy — max pooling, global average pooling, or both?

**Hints**:
- Start simple (2-3 conv blocks) and add complexity if needed
- Batch normalization helps training stability
- Dropout helps prevent overfitting
- Global average pooling before the classifier reduces parameter count

In [ ]:
# TODO: Define your CNN model
class VeggieCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Block 1: Input (3, 224, 224) -> Output (32, 112, 112)
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)

        # Block 2: Input (32, 112, 112) -> Output (64, 56, 56)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # Block 3: Input (64, 56, 56) -> Output (128, 28, 28)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # Global Average Pooling: Flattens the image to 1x1 pixels while keeping depth
        self.gap = nn.AdaptiveAvgPool2d(1)

        # Dropout to prevent overfitting
        self.dropout = nn.Dropout(0.3)

        # Final Fully Connected Layer (Classifier)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        # Apply Block 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))

        # Apply Block 2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        # Apply Block 3
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        # Global Average Pool and Flatten
        x = self.gap(x)
        x = x.view(x.size(0), -1)

        # Dropout and Classify
        x = self.dropout(x)
        x = self.fc(x)

        return x


scratch_model = VeggieCNN().to(device)
print(scratch_model)

# Print parameter count
total_params = sum(p.numel() for p in scratch_model.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
# TODO: Set up your training — choose optimizer, loss function, and hyperparameters
# Hyperparameters
NUM_EPOCHS = 15
LEARNING_RATE = 0.001

# 1. Loss Function (The "Judge")
criterion = nn.CrossEntropyLoss()

# 2. Optimizer (The "Trainer")
# We tell it which parameters to update and how fast to go
optimizer = optim.Adam(scratch_model.parameters(), lr=LEARNING_RATE)

# 3. Scheduler (The "Schedule")
# Every 7 steps (epochs), multiply the learning rate by 0.1 (make it 10x smaller)
scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

In [ ]:
from tqdm.auto import tqdm #just for loading bar to see progress

# TODO: Write your training loop
# Track train_losses, val_losses, train_accs, val_accs for plotting

# 1. Initialize the lists to track progress
train_losses, val_losses = [], []
train_accs, val_accs = [], []

print(f"Starting training on {device}...")

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Training phase ---
    scratch_model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Add a progress bar for the training batches
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}", unit="batch")

    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)

        # Optimization steps
        optimizer.zero_grad()
        outputs = scratch_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Update statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Update the loading bar with current loss
        progress_bar.set_postfix(loss=loss.item())

    # Calculate average training metrics
    train_loss = running_loss / len(train_loader.dataset)
    train_acc = 100 * correct / total

    # Step the scheduler
    scheduler.step()

    # --- Validation phase ---
    # Based on your error, we've removed 'device' from evaluate()
    # If it fails again, try: val_loss, val_acc = evaluate(scratch_model, val_loader)
    val_loss, val_acc = evaluate(scratch_model, val_loader, device)

    # Save metrics for plotting
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    # Print the summary for the epoch
    print(f"\n[Epoch {epoch}] Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%\n")

In [ ]:
# JUST FOR SAVING!!

import torch

# Save the model to a file
torch.save(scratch_model.state_dict(), 'veggie_model_weights.pth')
print("Model weights saved locally!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp -r /content/project2-nnimage-data-3950-team-1 /content/drive/MyDrive/

In [ ]:
# Plot your training curves
# plot_training_history(train_losses, val_losses, train_accs, val_accs)

In [ ]:
# Show per-class accuracy
# per_class_accuracy(scratch_model, val_loader, class_names, device)

---

## Part 2: Fine-Tune a Pretrained Model

Load a pretrained model (e.g., ResNet-18) and adapt it for veggie classification.

Steps:
1. Load the pretrained model with `models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)`
2. Replace the final classification layer for 10 classes
3. Choose a training strategy:
   - **Feature extraction**: Freeze all pretrained layers, train only the new head
   - **Fine-tuning**: Train the entire network with a lower learning rate for pretrained layers

**Hints**:
- Use a lower learning rate for pretrained layers than for the new head
- Data augmentation (random flips, rotations, color jitter) can help a lot
- A learning rate scheduler can improve final accuracy
- You can try other models: `resnet50`, `mobilenet_v3_small`, `efficientnet_b0`

In [ ]:
# TODO: Load and modify a pretrained model
# pretrained_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Replace the final layer:
# pretrained_model.fc = nn.Linear(pretrained_model.fc.in_features, 10)

# pretrained_model = pretrained_model.to(device)

In [ ]:
# TODO: Set up optimizer (consider differential learning rates)
# TODO: Train the model
# TODO: Plot results

---

## Compare Your Models

Fill in the table below with your results.

| | CNN from Scratch | Pretrained (Fine-tuned) |
|---|---|---|
| Parameters | ??? | ??? |
| Epochs trained | ??? | ??? |
| Best val accuracy | ???% | ???% |
| Training time | ??? | ??? |

---

## Save Your Best Model

Save your best model for submission. The file must be **under 400 MB** for Moodle.

In [ ]:
# Save your best model (pick whichever performed better)
# best_model = ...  # scratch_model or pretrained_model

# torch.save(best_model.state_dict(), "veggie_model.pth")

# Check file size
# import os
# size_mb = os.path.getsize("veggie_model.pth") / (1024 * 1024)
# print(f"Model size: {size_mb:.1f} MB")
# assert size_mb < 400, f"Model too large for Moodle! ({size_mb:.1f} MB > 400 MB)"

---

## Tips for Improving Accuracy

Here are some strategies to try (mention what you used in your explanatory note!):

**Data augmentation** — add to your training transforms:
- `transforms.RandomHorizontalFlip()`
- `transforms.RandomRotation(15)`
- `transforms.ColorJitter(brightness=0.2, contrast=0.2)`
- `transforms.RandomResizedCrop(224, scale=(0.8, 1.0))`

**Architecture choices**:
- Add batch normalization after each conv layer
- Use dropout (0.3-0.5) to reduce overfitting
- Try deeper or wider networks

**Training tricks**:
- Learning rate schedulers (`CosineAnnealingLR`, `ReduceLROnPlateau`)
- Weight decay in the optimizer (e.g., `weight_decay=1e-4`)
- Train for more epochs if not overfitting
- Use `AdamW` instead of `Adam`